In [1]:
import numpy as np
import pandas as pd
import warnings
import math
from datetime import timedelta
from glob import glob
import re

In [2]:
class Meta:
    def __init__(self, VarNames, fileUnits, fileDepth):
        self.VarNames = VarNames
        self.fileUnits = fileUnits
        self.fileDepth = fileDepth

In [3]:
class Prev():
    def __init__(self):
        pass

    def update(self, meta, dateData, numericData, uniquedatetime_Length):
        self.VarNames = meta.VarNames
        self.fileUnits = meta.fileUnits
        self.dateData = dateData
        self.numericData = numericData
        self.uniuquedatatime_Length = uniquedatetime_Length
        return self

In [4]:
# UNDERSTOOD
def readCSVFile(filepath):
    T = pd.read_csv(filepath)
    return T

In [5]:
# UNDERSTOOD
def unitConverter(fileUnits):
    if 'knots' in fileUnits:
        convertFactor = 0.51444448824222
    elif 'cm/sec' in fileUnits:
        convertFactor = 0.01
    else:
        warnings.warn('File defines water speed neither in knots nor cm/s. '
                      'Conversion factor of 1 is given by default. '
                      'Please Check file\'s units.')
        convertFactor = 1
    return convertFactor

In [6]:
# UNDERSTOOD
def extractData(T):
    dateData = pd.to_datetime(T.iloc[:, 0])
    numericaData = T.iloc[:, 1:]
    return dateData, numericaData

In [7]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation)
def extractMetaData(T, filename):
    VarNames = list(T.columns)

    speed_col  = next((name for name in VarNames if 'Speed' in name), None)
    if not speed_col:
        raise ValueError(f"{filename} does not contain 'Speed' column")

    # Extract units within the CSV file
    unit_match = re.search(r"Speed\s*[\(_]?([^\)]+)[\)]?", speed_col, re.IGNORECASE)
    fileUnits = unit_match.group(1).strip() if unit_match else ""

    # Extract depth from CSV file name
    depth_match = re.search(r"_(.*?)-", filename)
    fileDepth = depth_match.group(1) if depth_match else ""

    return Meta(VarNames, fileUnits, fileDepth)

In [8]:
# UNDERSTOOD
def valVarNames(current, previous, filePrev, fileCurr):
    if not current == previous:
        raise TypeError(f"Column mismatch between {filePrev}s and {fileCurr}s.")

In [9]:
# CHECK THIS CODE -- return variable mismatch
def initializeTimeSeries(dateData, numericData):
    dateDiff = []

    for i in range(len(dateData) - 1):
        dateDiff.append(dateData[i+1] - dateData[i])

        if not dateDiff[i] == dateDiff[0]:
            warnings.warn("There is uneven date interval within current dataset, and so first interval is chosen by default.")
            break

    dateInterval = dateDiff[0]

    dateOut = dateData
    dataOut = numericData
    dateOut_unique = dateData
    uniquedatetime_Length = len(dateData)

    return dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique

In [10]:
# CHECK THIS CODE
def processTimeSeries(prev, dateData, numericData, fileDepth, fileCurrName, filePrevName, dateInterval):
    if (
        dateData[0] > prev.dateData.iloc[-1]
        and abs(dateInterval - (dateData[0] - prev.dateData.iloc[-1])) < timedelta(seconds = 1)
        and fileDepth == prev.fileDepth
    ):
        uniquedatetime = np.unique(pd.concat([prev.dateData, dateData], axis=0))
        uniquedatetime_Length = len(uniquedatetime)

        if not uniquedatetime_Length == prev.uniquedatetime_Length:
            timeDiff = abs(uniquedatetime_Length - prev.uniquedatetime_Length)
            timeDiff_datetime = timeDiff * dateInterval
            warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}."
                          f"The previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}."
                          f"The current file ranges from {dateData[0]} to {dateData.iloc[-1]}."
                          f"This number of entries correspond to a time difference of {timeDiff_datetime}.")
            user_concatenate = input("Do you approve the concatenation of these datasets?"
                                     "Please write '1' to approve or '0' to disapprove: ")

            if user_concatenate == "1":
                dateOut_unique = uniquedatetime[0:uniquedatetime_Length]
            else:
                dateOut_unique = uniquedatetime[0:prev.uniquedatetime_Length]

        dateOut = uniquedatetime
        dataOut = pd.concat([prev.numericData, numericData], axis=0, ignore_index=True)

    elif dateData.equal(prev.dateData):
        dateOut = prev.dateData
        dataOut = numericData
    elif not dateData.equals(prev.dateData) and fileDepth == prev.fileDepth:
        warnings.warn(f"Date mismatch between files {filePrevName} and {fileCurrName}.")
        dateOut = dateData
        dataOut = numericData
    else:
        dateOut = dateData
        dataOut = numericData

    try:
        uniquedatetime_Length
    except NameError:
        uniquedatetime_Length = prev.uniquedatetime_Length

    try:
        dateOut_unique
    except NameError:
        dateOut_unique = prev.dateOut_unique

    return dateOut, dateOut_unique, dataOut, uniquedatetime_Length

In [11]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation)
def extractSiteID(filenames):
    extraced_IDs = []

    for name in filenames:
        match = re.match(r'^([a-zA-Z]+)(\d+)', name)
        if match:
            extraced_IDs.append(match.group(1) + match.group(2))
        else:
            extraced_IDs.append("")
    if not extraced_IDs or any(s != extraced_IDs[0] for s in extraced_IDs):
        warnings.warn('There is a mismatch of site ID within provided data.')
        print('User-defined ID requested for plotting: ')
        return inputID()
    return extraced_IDs[0]

In [12]:
# UNDERSTOOD
def inputID():
    user_decision = input('Do you want to continue by defining the site ID? (Y/N): ')

    if user_decision == 'N':
        print('Exiting from FolderRead.'
              'recommendation to revise data in files.')
        return

    siteID = input('\n Please define the site ID to appear in plots (e.g. LIS1001): ')
    print('Continuing with user')
    return siteID

In [13]:
# UNDERSTOOD -- FIXED CODE FOR EFFICIENCY (regex implementation, simplified decimal creation)
def findDepth(filenames, files_num):
    fileDepths = []
    depthUnits = []

    pattern = re.compile(r'_(\d+)[a-zA-Z]+(\d+)([a-zA-Z]+)-?')

    for filename in filenames:
        match = pattern.search(filename)
        if not match:
            raise ValueError(
                'waterDepth:incorrectFormat',
                f"Error in file format for '{filename}'. \n"
                'File format must list same units after site ID.\n'
                "Acceptable formats are the following: 'LIS1001_05m76cm' or 'LIS1001_18ft09df'."
            )

        integer_part, decimal_part, unit = match.groups()

        depth_value = float(f"{integer_part}.{decimal_part}")

        fileDepths.append(depth_value)
        depthUnits.append(unit)

    if len(set(depthUnits)) > 1:
        raise ValueError(
            'waterDepth:incorrectFormat',
            'Error in file format.\nFile format must list same units across all files.'
        )

    depth_units = depthUnits[0] if depthUnits else ""
    waterDepth = np.double(np.unique(fileDepths))

    return waterDepth, depth_units


In [14]:
def alignDataLengths(dataCells, targetLength):
    pass

In [15]:
def FolderReadCSV(folderpath):
    pass